# Livrable 1 — Modélisation du problème de tournées de livraison

**Projet :** CesiCDP — réponse à l’appel à manifestation d’intérêt de l’ADEME  
**Problème retenu :** CVRP (Capacitated Vehicle Routing Problem), avec flotte homogène et dépôt unique.

> Ce notebook constitue le livrable de modélisation demandé au premier point de suivi. Il formalise le problème et sa complexité, puis propose une première formulation PLNE. Il ne traite pas du choix d’une méthode de résolution ni de son implémentation.

## 1. Contexte et périmètre

CesiCDP souhaite optimiser des tournées de livraison afin de réduire les déplacements et leur impact environnemental. Une flotte de camions part d’un dépôt, dessert des clients ayant chacun une demande, puis revient au dépôt. Le coût d’un trajet entre deux lieux représente ici une distance ou une durée de trajet connue.

L’énoncé général évoque un trafic qui varie selon les tranches horaires et une version avancée avec fenêtres de livraison. Pour disposer d’un périmètre cohérent avec le **niveau B attendu dans les critères du projet**, ce livrable retient le CVRP : capacité identique pour tous les véhicules, demandes clients, dépôt unique et coût de trajet fixe.

L’objectif est de minimiser la somme des coûts des arcs parcourus. Chaque client doit être visité exactement une fois, la charge de chaque camion ne doit pas dépasser sa capacité et chaque tournée doit partir du dépôt et y revenir.

## 2. Hypothèses et données

On considère un graphe complet orienté. Le coût direct entre deux sommets peut être la distance routière la plus courte pré-calculée sur le réseau réel. Les coûts sont non négatifs et fixes pendant la tournée. Chaque véhicule utilisé dessert au moins un client. Le nombre de véhicules disponibles est une borne supérieure : des véhicules peuvent rester inutilisés.

### Ensembles et paramètres

- $C=\{1,\ldots,n\}$ : ensemble des clients ; $0$ : dépôt ; $V=C\cup\{0\}$.
- $K=\{1,\ldots,m\}$ : véhicules disponibles.
- $q_i>0$ : demande du client $i\in C$.
- $Q>0$ : capacité identique de chaque véhicule.
- $c_{ij}\geq 0$ : coût de déplacement de $i$ vers $j$, pour $i,j\in V$, $i\ne j$.
- Condition de faisabilité globale : $\sum_{i\in C}q_i\leq mQ$ (condition nécessaire, mais pas suffisante à elle seule).

Si les données de départ décrivent un réseau routier non complet, $c_{ij}$ peut être calculé comme le coût du plus court chemin entre les lieux $i$ et $j$. Il faut ensuite documenter l’unité retenue (km ou minutes) et s’assurer que les coûts correspondent au même scénario de trafic.

## 3. Définition formelle

**Entrée :** un ensemble de clients avec leurs demandes, un dépôt, au plus $m$ véhicules de capacité $Q$, et les coûts $c_{ij}$.

**Solution réalisable :** un ensemble d’au plus $m$ circuits partant du dépôt et y revenant, tel que chaque client apparaisse dans exactement un circuit et que la demande totale affectée à chaque circuit soit au plus $Q$.

**Objectif :** trouver une solution réalisable minimisant le coût total $\sum c_{ij}$ des arcs utilisés.

La version de décision associée ajoute à l’entrée un seuil $B$ et demande : « existe-t-il une tournée réalisable dont le coût total est inférieur ou égal à $B$ ? »

## 4. Complexité

### Appartenance à NP (version de décision)

Un certificat est la liste ordonnée des clients de chaque tournée. Sa taille est polynomiale en $n$. Un vérificateur peut contrôler en temps polynomial que chaque client apparaît une fois, que chaque tournée commence et finit au dépôt, que la charge de chaque tournée ne dépasse pas $Q$, que le nombre de tournées ne dépasse pas $m$, puis additionner les coûts et vérifier le seuil $B$. Le problème de décision appartient donc à NP.

### NP-difficulté par réduction depuis TSP décision

On réduit le problème du voyageur de commerce (TSP) décision, connu NP-complet, au CVRP décision. Soit une instance TSP sur $n$ villes, avec coûts $c_{ij}$ et seuil $B$. Construisons une instance CVRP avec le dépôt comme ville de départ, les autres villes comme clients, une demande unitaire par client, un seul véhicule ($m=1$) de capacité $Q=n-1$, et les mêmes coûts d’arcs.

Il existe un circuit TSP de coût au plus $B$ si et seulement s’il existe une tournée CVRP de coût au plus $B$ : avec un véhicule et une capacité suffisante, la tournée doit visiter chaque client une fois et revenir au dépôt, exactement comme le circuit TSP. La transformation est polynomiale. Le CVRP décision est donc NP-difficile. Comme il appartient à NP, il est NP-complet. Le problème d’optimisation CVRP est ainsi NP-difficile.

Cette preuve établit la difficulté dans le cas général ; elle ne signifie pas que toute instance pratique est difficile au même degré.

## 5. Première formulation en Programme Linéaire en Nombres Entiers (PLNE)

Cette formulation utilise des variables **indexées par véhicule**. Cela permet de savoir non seulement quels arcs sont utilisés, mais aussi **quel camion** les emprunte. En contrepartie, il faut davantage de variables et de contraintes qu'avec une formulation qui ne distingue pas les véhicules. Elle s’inscrit dans le cadre général de la programmation linéaire présenté par [Techniques de l’ingénieur — Programmation linéaire](https://www.techniques-ingenieur.fr/ressources-documentaires/sciences-fondamentales-th8/mathematiques-ti052/programmation-lineaire-af1254) [2].

On note :
- $C=\{1,\ldots,n\}$ : ensemble des clients ;
- $V=C\cup\{0\}$ : clients et dépôt ;
- $K=\{1,\ldots,m\}$ : véhicules disponibles ;
- $A=\{(i,j)\in V^2:i\ne j\}$ : arcs possibles.

### 5.1 Variables de décision

- $x_{ijk}\in\{0,1\}$ : vaut 1 si le véhicule $k$ se déplace directement de $i$ vers $j$, et 0 sinon.
- $y_{ik}\in\{0,1\}$ : vaut 1 si le client $i$ est affecté au véhicule $k$.
- $z_k\in\{0,1\}$ : vaut 1 si le véhicule $k$ est utilisé.
- $u_{ik}\geq 0$ : quantité cumulée livrée par le véhicule $k$ jusqu'à sa visite du client $i$. Cette variable auxiliaire sert à modéliser la capacité et à éliminer les sous-tours.

**Pourquoi quatre familles de variables ?** Elles ne représentent pas la même décision : $x$ décrit les déplacements, $y$ l'affectation des clients, $z$ l'utilisation des camions et $u$ la progression de la charge cumulée. Une variable n'a pas besoin d'apparaître dans la fonction objectif pour être utile : elle peut servir uniquement aux contraintes.

### 5.2 Fonction objectif

On minimise le coût total des arcs empruntés par tous les véhicules :

$$\min Z=\sum_{k\in K}\sum_{(i,j)\in A}c_{ij}x_{ijk}$$

Ici, $c_{ij}$ représente la distance (en km) ou la durée (en minutes) entre $i$ et $j$. Comme $x_{ijk}=1$ uniquement si le véhicule $k$ emprunte cet arc, la somme additionne les coûts de toutes les tournées.

### 5.3 Contraintes

#### A. Chaque client est affecté à un seul véhicule

$$\sum_{k\in K}y_{ik}=1 \qquad \forall i\in C$$

Pour chaque client $i$, exactement un véhicule est choisi. Cette contrainte porte sur **l'affectation** des clients, pas directement sur les arcs.

#### B. Un client affecté au véhicule doit avoir une entrée et une sortie

$$\sum_{j\in V\setminus\{i\}}x_{ijk}=y_{ik}
\qquad \forall i\in C,\ k\in K$$

$$\sum_{j\in V\setminus\{i\}}x_{jik}=y_{ik}
\qquad \forall i\in C,\ k\in K$$

Si $y_{ik}=1$, le véhicule $k$ a exactement un arc sortant du client $i$ et un arc entrant dans ce client. Si $y_{ik}=0$, ces sommes valent 0 : ce véhicule ne visite pas ce client.

#### C. Chaque véhicule utilisé part du dépôt et y revient

$$\sum_{j\in C}x_{0jk}=z_k \qquad \forall k\in K$$

$$\sum_{i\in C}x_{i0k}=z_k \qquad \forall k\in K$$

Si $z_k=1$, le véhicule $k$ a un départ du dépôt et un retour au dépôt. Si $z_k=0$, il n'effectue aucun de ces déplacements.

#### D. Relier l'affectation à l'utilisation du véhicule

$$y_{ik}\leq z_k \qquad \forall i\in C,\ k\in K$$

Un véhicule inutilisé ($z_k=0$) ne peut pas se voir affecter de client. Cette contrainte rend explicite le lien entre les variables $y$ et $z$.

#### E. Respecter la capacité du véhicule

$$\sum_{i\in C}q_i y_{ik}\leq Qz_k \qquad \forall k\in K$$

La demande totale des clients affectés à un véhicule ne dépasse pas sa capacité $Q$. Si le véhicule n'est pas utilisé, $z_k=0$ et la contrainte impose qu'aucun client ne lui soit affecté.

#### F. Cohérence de charge et élimination des sous-tours

On borne d'abord la charge cumulée :

$$q_i y_{ik}\leq u_{ik}\leq Qy_{ik}
\qquad \forall i\in C,\ k\in K$$

Si le client $i$ n'est pas affecté au véhicule $k$, alors $y_{ik}=0$ et donc $u_{ik}=0$. S'il lui est affecté, la charge cumulée est au moins égale à la demande de ce client et ne dépasse pas $Q$.

Ensuite, pour deux clients distincts $i$ et $j$ :

$$u_{jk}\geq u_{ik}+q_j-Q(1-x_{ijk})
\qquad \forall k\in K,\ i,j\in C,\ i\ne j$$

**Si l'arc $i\to j$ est emprunté**, $x_{ijk}=1$ et la contrainte devient :

$$u_{jk}\geq u_{ik}+q_j$$

La quantité cumulée augmente donc au moins de la demande $q_j$ du client suivant.

**Si l'arc $i\to j$ n'est pas emprunté**, $x_{ijk}=0$ et la contrainte devient :

$$u_{jk}\geq u_{ik}+q_j-Q$$

Comme $u_{ik}\leq Qy_{ik}\leq Q$ et $u_{jk}\geq 0$, cette inégalité est suffisamment relâchée pour ne pas imposer la progression de charge associée à l'arc non utilisé.

Cette progression empêche un sous-tour formé uniquement de clients : si un cycle de clients était utilisé, la charge cumulée devrait augmenter d'une quantité strictement positive à chaque arc, puis revenir à son point de départ, ce qui est impossible.

#### G. Domaine des variables

$$x_{ijk},y_{ik},z_k\in\{0,1\},\qquad u_{ik}\geq 0$$

Les variables $x$, $y$ et $z$ sont binaires, tandis que $u$ est continue. Elle n'a pas besoin d'être entière pour que cette formulation fonctionne.

### 5.4 Pourquoi y a-t-il plus de contraintes que dans une formulation plus simple ?

Le nombre de contraintes dépend de la façon dont on définit les variables.

- **Formulation sans indice de véhicule** : $x_{ij}$ indique simplement si l'arc $(i,j)$ est utilisé par l'une des tournées. Le modèle distingue moins explicitement les camions.
- **Formulation avec indice de véhicule** : $x_{ijk}$ identifie le camion qui emprunte l'arc. Il faut alors aussi définir l'affectation $y_{ik}$ et l'utilisation $z_k$, puis relier ces variables entre elles.

Les contraintes supplémentaires ne sont donc pas arbitraires : elles garantissent la cohérence entre **le client affecté**, **les arcs empruntés**, **le camion utilisé**, **la capacité** et **la structure des tournées**. Les formulations ne sont pas directement interchangeables sans adapter leurs contraintes.

Remarque : la contrainte de capacité agrégée en $y$ et les bornes de $u$ sont en partie redondantes sur la capacité, mais elles ont des rôles explicatifs différents. La première limite la demande totale affectée au camion ; les secondes permettent aussi d'activer les contraintes de progression de charge qui éliminent les sous-tours.

La formulation suppose une flotte homogène, un dépôt unique, des demandes strictement positives et une livraison de chaque client en une seule visite. La ressource pédagogique [Programmation linéaire et PuLP](https://scenari.cesi.fr/scenari/cas/Diffusion_FISE_INFO_A3_Algorithmique_et_optimisation_26_27_ETU/res/PRogLineMaire_Et_Pulp_V1.pdf) [3] pourra servir de transition vers l’implémentation de ce modèle dans un solveur ; cette étape est volontairement hors du périmètre de ce livrable.

## 6. Limites et prolongements de la modélisation

Cette première formulation suppose des coûts d’arcs fixes et n’inclut pas de fenêtres horaires, de durée maximale de tournée, de coûts d’émission séparés, ni de trafic dépendant de l’heure. Pour traiter le CVRPTW, il faudrait ajouter des variables de temps d’arrivée et des contraintes de fenêtres, d’attente et de propagation des temps. Pour le trafic variable, le coût d’un arc dépendrait de l’heure de départ sur cet arc ; une formulation devrait représenter ces temps et la fonction de coût temporelle. Ces choix seront à préciser avant d’étendre le PLNE.

Les données cartographiques doivent aussi être préparées : sélection des clients et du dépôt, demandes, matrice de coûts, unités et hypothèses de circulation. La formulation fournit une base vérifiable et implémentable ; elle ne constitue pas encore un solveur.

## 7. Références

### Ressources fournies

[P versus NP : exemple dans un réseau social](https://www.youtube.com/watch?v=rkDlTQFucBM), Rachid Guerraoui, vidéo YouTube. — **Complexité et distinction P/NP**.

[Programmation linéaire](https://www.techniques-ingenieur.fr/ressources-documentaires/sciences-fondamentales-th8/mathematiques-ti052/programmation-lineaire-af1254), Techniques de l’ingénieur. — **Cadre de la programmation linéaire**.

[Programmation linéaire et PuLP](https://scenari.cesi.fr/scenari/cas/Diffusion_FISE_INFO_A3_Algorithmique_et_optimisation_26_27_ETU/res/PRogLineMaire_Et_Pulp_V1.pdf), support pédagogique CESI. — **Passage du modèle à l’implémentation**.

[La modélisation — Recherche opérationnelle](https://www.youtube.com/watch?v=Zy-iTwA9tfU), vidéo YouTube. — **Démarche de modélisation**.